In [11]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [3]:
df_features = pd.read_csv("../data/processed/cleaned_waitlist.csv")
df_features.head()

,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,INIT_AGE,DIALYSIS_DATE,...,WL_ID_CODE,PREV_TX,DIAG_KI,MULTIORG,LISTING_CTR_CODE,outcome,event_adverse,event_transplant,censored,days_to_event
0,Y,NaN,F,B,31.63,2080.0,4099,0.0,53,2018-03-30,...,1526706,Unknown,-1.0,N,13609,died,1,0,0,287.0
1,Y,NaN,M,A,30.04,2070.0,4099,0.0,56,2017-08-16,...,1519605,Unknown,-1.0,N,6975,removed_administrative,0,0,0,2322.0
2,N,NaN,F,O,32.85,2070.0,4099,0.0,47,Not on dialysis,...,1535094,Unknown,-1.0,N,19716,died,1,0,0,604.0
3,Y,NaN,M,A,20.00,2090.0,4099,0.0,61,2019-01-05,...,1527782,Unknown,-1.0,N,8587,removed_too_sick,1,0,0,909.0
4,Y,NaN,M,AB,23.30,2070.0,4010,0.0,61,2019-01-10,...,1517273,Unknown,-1.0,N,18352,removed_too_sick,1,0,0,231.0


In [4]:
# Select only the recommended features per data_dictionary.md
feature_cols = [
    'ON_DIALYSIS', 'A2A2B_ELIGIBILITY', 'GENDER', 'ABO', 'BMI_TCR',
    'FUNC_STAT_TCR', 'INIT_STAT', 'INIT_CPRA', 'INIT_AGE',
    'DIALYSIS_DATE', 'INIT_DATE', 'ETHCAT', 'REGION'
]

df_model = df_features[feature_cols].copy()
df_model.head()

,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,INIT_AGE,DIALYSIS_DATE,INIT_DATE,ETHCAT,REGION
0,Y,NaN,F,B,31.63,2080.0,4099,0.0,53,2018-03-30,2020-03-25,2,8
1,Y,NaN,M,A,30.04,2070.0,4099,0.0,56,2017-08-16,2020-02-14,1,5
2,N,NaN,F,O,32.85,2070.0,4099,0.0,47,Not on dialysis,2020-05-27,1,11
3,Y,NaN,M,A,20.00,2090.0,4099,0.0,61,2019-01-05,2020-04-02,5,7
4,Y,NaN,M,AB,23.30,2070.0,4010,0.0,61,2019-01-10,2020-02-05,2,7


In [5]:
categorical_cols = [
    'ON_DIALYSIS', 'GENDER', 'ABO', 'A2A2B_ELIGIBILITY',
    'INIT_STAT', 'ETHCAT', 'REGION'
]

numerical_cols = ['BMI_TCR', 'INIT_CPRA', 'INIT_AGE']

# FUNC_STAT_TCR needs special decoding (per data_dictionary.md) not simple encoding or scaling
# DIALYSIS_DATE / INIT_DATE need date handling (per data_dictionary.md) not simple encoding or scaling

In [6]:
#Caps BMI at a maximum of 80, there was a data entry error
df_model.loc[df_model['BMI_TCR'] > 80, 'BMI_TCR'] = df_model['BMI_TCR'].median()
df_model['BMI_TCR'].describe()

count    494803.000000
mean         28.823687
std           5.809189
min           0.180000
25%          24.660000
50%          28.530000
75%          32.820000
max          78.980000
Name: BMI_TCR, dtype: float64

In [7]:
#encode categorical data
new_df = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)
new_df.head()

,BMI_TCR,FUNC_STAT_TCR,INIT_CPRA,INIT_AGE,DIALYSIS_DATE,INIT_DATE,ON_DIALYSIS_Y,GENDER_M,ABO_A1,ABO_A1B,...,REGION_2,REGION_3,REGION_4,REGION_5,REGION_6,REGION_7,REGION_8,REGION_9,REGION_10,REGION_11
0,31.63,2080.0,0.0,53,2018-03-30,2020-03-25,True,False,False,False,...,False,False,False,False,False,False,True,False,False,False
1,30.04,2070.0,0.0,56,2017-08-16,2020-02-14,True,True,False,False,...,False,False,False,True,False,False,False,False,False,False
2,32.85,2070.0,0.0,47,Not on dialysis,2020-05-27,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3,20.00,2090.0,0.0,61,2019-01-05,2020-04-02,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False
4,23.30,2070.0,0.0,61,2019-01-10,2020-02-05,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False


In [8]:
#Check INIT_CPRA range/outliers and most frequent values (which is 0.00)
df_model['INIT_CPRA'].describe()
df_model['INIT_CPRA'].value_counts().head(10)

INIT_CPRA
0.00      420160
0.03         904
16.56        586
0.26         583
100.00       568
99.99        542
56.09        500
2.32         499
50.01        496
0.02         474
Name: count, dtype: int64

In [12]:
#normalize numerical data
scaler = StandardScaler()
new_df[numerical_cols] = scaler.fit_transform(new_df[numerical_cols])
new_df[numerical_cols].head()

,BMI_TCR,INIT_CPRA,INIT_AGE
0,0.483082,-0.32562,0.077090
1,0.209378,-0.32562,0.282551
2,0.693094,-0.32562,-0.333834
3,-1.518920,-0.32562,0.624987
4,-0.950854,-0.32562,0.624987


In [13]:
new_df.to_csv("../data/processed/task3_encoded_scaled.csv", index=False)